In [7]:
import pandas as pd
import spacy
from pathlib import Path
from sklearn.model_selection import train_test_split
from spacy.training.example import Example
from spacy.util import filter_spans
import ast

In [8]:
#Identify base directory to ensure portability
BASE_DIR = Path.cwd().resolve().parent
MODELS_DIR = BASE_DIR / "models"
DATA = BASE_DIR / "data/processed/distilled.csv"

In [9]:
df = pd.read_csv(DATA)
display(df)

,Unnamed: 0,text,propaganda
0,0,Outrage as Donald Trump suggests injecting dis...,"[{'span': [0, 6], 'technique': 'Loaded_Languag..."
1,1,The senator's vile betrayal of working familie...,"[{'span': [14, 18], 'technique': 'Loaded_Langu..."
2,2,Brave freedom fighters resist the tyrannical o...,"[{'span': [0, 5], 'technique': 'Loaded_Languag..."
3,3,The corrupt elites are bleeding this country d...,"[{'span': [4, 10], 'technique': 'Loaded_Langua..."
4,4,A catastrophic failure of leadership has plung...,"[{'span': [2, 13], 'technique': 'Loaded_Langua..."
...,...,...,...
4501,5852,Altered Election Documents Tied To Florida Dem...,"[{'span': [86, 101], 'technique': 'Loaded_Lang..."
4502,5853,Migrant Caravan Reach Border & Climb Atop Fenc...,"[{'span': [31, 62], 'technique': 'Loaded_Langu..."
4503,5854,Guardian ups its vilification of Julian Assang...,"[{'span': [17, 29], 'technique': 'Loaded_Langu..."
4504,5855,This Guardian Fake News Story Proves That The ...,"[{'span': [0, 68], 'technique': 'Doubt'}, {'sp..."


In [10]:
def sledgehammer_clean(val):
    """Recursively unpacks strings until a list/dict is found."""
    if isinstance(val, (list, dict)):
        return val
    if not val or pd.isna(val) or val == 'nan':
        return []
    try:
        #Attempt to parse the string
        unpacked = ast.literal_eval(val)
        #If the result is STILL a string, go deeper
        return sledgehammer_clean(unpacked)
    except (ValueError, SyntaxError):
        return []

df['propaganda'] = df['propaganda'].apply(sledgehammer_clean)
df = df[df['propaganda'].map(lambda x: len(x) > 0 if isinstance(x, list) else False)].copy()
df.head()

,Unnamed: 0,text,propaganda
0,0,Outrage as Donald Trump suggests injecting dis...,"[{'span': [0, 6], 'technique': 'Loaded_Languag..."
1,1,The senator's vile betrayal of working familie...,"[{'span': [14, 18], 'technique': 'Loaded_Langu..."
2,2,Brave freedom fighters resist the tyrannical o...,"[{'span': [0, 5], 'technique': 'Loaded_Languag..."
3,3,The corrupt elites are bleeding this country d...,"[{'span': [4, 10], 'technique': 'Loaded_Langua..."
4,4,A catastrophic failure of leadership has plung...,"[{'span': [2, 13], 'technique': 'Loaded_Langua..."


In [11]:
TECH_MAP = {
    "Red_Herring": "Whataboutism_Straw_Men_Red_Herring",
    "Minimisation": "Exaggeration_Minimisation",
    "Exaggeration": "Exaggeration_Minimisation",
    "Straw_Men": "Whataboutism_Straw_Men_Red_Herring",
    "Whataboutism": "Whataboutism_Straw_Men_Red_Herring"}

In [12]:
def get_final_spacy_data(data_frame):
    nlp = spacy.blank("en")
    formatted = []

    for _, row in data_frame.iterrows():
        text = str(row['text'])
        doc = nlp.make_doc(text)
        spans = row['propaganda']

        entities = []
        for s in spans:
            try:
                start, end = s['span']
                #Standardize to the 14 categories
                label = TECH_MAP.get(s['technique'], s['technique'])

                #FIX [W030]: Align offsets to valid token boundaries
                span = doc.char_span(start, end, label=label, alignment_mode="expand")
                if span:
                    entities.append(span)
            except:
                continue

        #FIX [E103]: Filter out overlaps.
        #filter_spans keeps the longest span if two overlap
        filtered_entities = filter_spans(entities)

        #Convert back to the tuple format spaCy needs for training
        final_ents = [(ent.start_char, ent.end_char, ent.label_) for ent in filtered_entities]
        formatted.append((text, {"entities": final_ents}))

    return formatted

df = get_final_spacy_data(df)

In [14]:
#Split data
train_df, test_df = train_test_split(df, test_size=0.2, random_state=42)
train_df[:2]

[('This candidate has a brain the size of a planet and a heart of pure gold.',
  {'entities': [(21, 51, 'Exaggeration_Minimisation')]}),
 ('Stalin also believed in strict gun control for his citizens.',
  {'entities': []})]

In [16]:
valid_rows = sum(1 for text, annot in train_df if len(annot['entities']) > 0)
print(f"Success! Found {valid_rows} rows with entities out of {len(train_df)}")

Success! Found 3281 rows with entities out of 3604


In [ ]:
#Train lightweight NER model
nlp = spacy.blank("en")
ner = nlp.add_pipe("ner")

In [ ]:
#Add all labels found in training
for _, annotations in train_data:
    for ent in annotations.get("entities"):
        ner.add_label(ent[2])

In [ ]:
optimizer = nlp.begin_training()

In [ ]:
#Training loop
for i in range(15):
    losses = {}
    for text, annotations in train_data:
        doc = nlp.make_doc(text)
        example = Example.from_dict(doc, annotations)
        nlp.update([example], drop=0.2, losses=losses)
    print(f"Epoch {i} - Loss: {losses['ner']:.4f}")

In [ ]:
def predict(text):
    doc = nlp(text)
    return [{"span": [ent.start_char, ent.end_char],
             "technique": ent.label_} for ent in doc.ents]

In [ ]:
#Test on a piece of the test set
sample = test_df.iloc[0]['text']
print(f"\nText: {sample}")
print(f"Prediction: {predict(sample)}")